# Preprocesamiento

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / "src"))
from preprocess import *

GUARDAR = False   # True: escribe data/clean/df_total.xlsx
VERSIONAR = False  # True: además lo registra en data/logs/data_log.json
OPTIMIZAR = False   # True: corre el estudio de Optuna de selección de variables (tarda unos minutos)
GUARDAR_FINAL = False  # True: escribe data/processed/dataset_final.csv + checksum y linaje en data/logs
RNG = 42

## Crear el dataframe

### Juntar las hojas

In [2]:
partes = [procesar_excel(ruta, crit_kpi="mean") for ruta in RAW_DEFAULT]
for ruta, parte in zip(RAW_DEFAULT, partes):
    print(f"{ruta.name}: {parte.shape} | {parte.index.min().date()} -> {parte.index.max().date()}")

Totalizadores Planta - 2021_2023.xlsx: (594, 396) | 2021-07-02 -> 2023-03-05
Totalizadores Planta 2020_2022.xlsx: (604, 315) | 2020-07-01 -> 2022-03-15
Totalizadores Planta 2022_2023.xlsx: (469, 396) | 2022-07-01 -> 2023-10-25


### Juntar los Excel


In [3]:
df = juntar_datasets(partes)
df.shape

(1178, 413)

### Columnas vacías y formatear


In [4]:
df, eliminadas = borrar_columnas_vacias(df, min_nulos=1100, umbral_varianza=10)
df = convertir_object_a_numerico(df)
print(f"Columnas eliminadas: {len(eliminadas)} | shape: {df.shape}")

Columnas eliminadas: 99 | shape: (1178, 314)


### Completar los días faltantes



In [5]:
dias_antes = len(df)
df = completar_dias(df)
print(f"Días agregados: {len(df) - dias_antes} | días consecutivos: "
      f"{(df[COL_FECHA].diff().dropna() == pd.Timedelta(days=1)).all()}")

Días agregados: 34 | días consecutivos: True


### Agregar features: clima y mes 
- Temperatura y humedad: media, máx, mín
- Mes: `mes_sin` / `mes_cos`

Las `COLS_SIN_LIMITES` no las tocan los pasos de inválidos y outliers.

In [6]:
df = agregar_clima(df)
df = agregar_mes_ciclico(df)
df[[COL_FECHA, *COLS_SIN_LIMITES]].describe().round(2)

,fecha,temp media (°c),temp max (°c),temp min (°c),humedad media (%),humedad max (%),humedad min (%),mes_sin,mes_cos
count,1212,1212.00,1212.00,1212.00,1212.00,1212.00,1212.00,1212.00,1212.00
mean,2022-02-26 12:00:00,17.67,24.30,11.87,55.57,78.45,33.39,-0.06,-0.06
min,2020-07-01 00:00:00,3.20,6.20,-4.30,12.00,19.00,4.00,-1.00,-1.00
25%,2021-04-29 18:00:00,12.00,18.20,6.70,41.00,65.00,19.00,-0.87,-0.87
50%,2022-02-26 12:00:00,17.10,24.40,11.40,56.00,86.00,31.00,0.00,-0.00
75%,2022-12-26 06:00:00,23.42,30.80,17.10,72.00,95.00,45.00,0.50,0.50
max,2023-10-25 00:00:00,34.80,44.00,26.10,98.00,100.00,89.00,1.00,1.00
std,NaN,6.83,7.71,6.52,19.78,20.24,17.87,0.70,0.71


### Variable objetivo

In [7]:
df = agregar_target(df)
df[[COL_FECHA, "frio (kw)", TARGET]].tail()

,fecha,frio (kw),frio_mañana (kw)
1207,2023-10-21,9481.0,11587.0
1208,2023-10-22,11587.0,10123.0
1209,2023-10-23,10123.0,10360.0
1210,2023-10-24,10360.0,10568.0
1211,2023-10-25,10568.0,NaN


## Guardar y versionar
Con `versionar=True` registra el hash del archivo en `data_log.json`.

In [8]:
if GUARDAR:
    df.to_excel(RUTA_LIMPIO, index=False)
    if VERSIONAR:
        registrar_version(RUTA_LIMPIO, ["preprocesamiento.ipynb"], comentario="...")

# equivalente en una línea:
# df = construir_dataset(guardar=RUTA_LIMPIO, versionar=True, comentario="...")
# o, si ya está construido:
# df = cargar_dataset_limpio()

## Variables derivadas

No se guardan en `df_total.xlsx`: se agregan después de cargar, así se pueden probar combinaciones cambiando `ConfigDerivadas` sin reconstruir desde los crudos.

- **Lags** (`lag1`, `lag2`, `lag7`) y **medias móviles** (`mm3`, `mm7`, `mm14`) de las `cols_temporales`: 
- **Ratios** entre consumos (por ejemplo `frio (kw) / planta (kw)`); si el denominador es 0 -> NaN.
- **Cumplimiento de meta**: `cumpl X` = real por hl / meta (1 = en meta, > 1 = por encima).
- **Interacciones**: productos entre variables principales.

In [9]:
cfg_derivadas = ConfigDerivadas()  # p. ej. ConfigDerivadas(lags=[1, 7], interacciones=[])
df = agregar_derivadas(df, cfg_derivadas, verbose=True)
df[[c for c in df.columns if c.startswith("cumpl ee")]].describe().round(2)

Variables derivadas agregadas: 69 | metas sin columna real: ['meta agua elab', 'meta agua bodega', 'meta agua envas', 'meta agua linea 4', 'meta ee elaboracion', 'meta ee bodega', 'meta ee cocina', 'meta ee linea 5', 'meta ee aire', 'meta ee co2', 'meta ee caldera', 'meta ee eflu', 'meta aire envasado']


,cumpl ee planta,cumpl ee envasado,cumpl ee linea 2,cumpl ee linea 3,cumpl ee linea 4,cumpl ee servicios,cumpl ee sala maq,cumpl ee frio,cumpl ee agua,cumpl ee resto serv,cumpl ee resto planta
count,825.00,8.250000e+02,735.00,712.00,726.00,825.00,825.00,825.00,825.00,825.00,8.250000e+02
mean,1.93,1.426766e+09,2.29,1.46,1.48,1.61,1.16,1.47,0.43,1.95,-1.374201e+10
std,7.28,2.922715e+10,3.30,2.25,3.47,2.96,2.74,2.55,42.35,13.74,2.815036e+11
min,-0.17,0.000000e+00,-0.04,0.00,0.00,-1.19,-0.01,0.00,-211.16,-218.31,-7.410066e+12
25%,0.94,7.300000e-01,0.98,0.91,0.87,0.90,0.60,0.84,0.36,1.10,2.100000e+00
50%,1.10,9.500000e-01,1.52,1.08,0.98,1.12,0.75,1.05,0.50,1.53,2.840000e+00
75%,1.44,1.230000e+00,2.13,1.32,1.11,1.42,0.94,1.34,0.75,2.21,4.060000e+00
max,193.79,7.693511e+11,31.16,29.11,45.28,59.02,57.85,49.28,989.27,150.83,6.864400e+03


## Clase Preprocess

### Configuración del preprocesamiento

| Paso | Config | Opciones |
|---|---|---|
| Inválidos | `ConfigInvalidos` | `no_negativas` (`"ee"` por default), `max_abs`, `rangos`; acción `nan` / `clip` / `delete` |
| Outliers | `ConfigOutliers` | `iqr` / `zscore_mod` / `percentil`; acción `nan` / `clip` / `delete`; `excluir` |
| Transformación | `ConfigTransformer` | `yeo-johnson` / `boxcox` / `log` / `sqrt` |
| Escalado | `ConfigScaler` | `standard` / `robust` / `minmax` |
| Faltantes | `ConfigFaltantes` | `mean` / `median` / `interpolate` / `knn` |
| Split | `ConfigSplit` | `test_size` (corte temporal) |

A todos se les puede pasar `None` para desactivarlos; tener cuidado con los metodos de `delete` porque borran días.

In [10]:
pp = Preprocess(
    invalidos=ConfigInvalidos(accion="nan"),
    outliers=ConfigOutliers(metodo="iqr", accion="nan"),
    transformer=ConfigTransformer(metodo="yeo-johnson"),
    scaler=ConfigScaler(metodo="robust"),
    faltantes=ConfigFaltantes(estrategia="knn", n_vecinos=5),
    split=ConfigSplit(test_size=0.15, val_size=0.15),
    verbose=True,
)

### Separación Train / validación / test 

Corte por fecha (70/15/15): val y test son los últimos días, para no predecir el pasado con el futuro.

In [11]:
train, val, test = pp.train_test_split(df)
for nombre, parte in {"train": train, "val": val, "test": test}.items():
    print(f"{nombre:5}: {parte.shape} {parte[COL_FECHA].min().date()} -> {parte[COL_FECHA].max().date()}")

train: (848, 392) 2020-07-01 -> 2022-10-26
val  : (182, 392) 2022-10-27 -> 2023-04-26
test : (182, 392) 2023-04-27 -> 2023-10-25


### Correr el preprocess


In [12]:
train_pp = pp.fit_transform(train)
val_pp = pp.transform(val, nombre="val")
test_pp = pp.transform(test, nombre="test")

X_train, y_train = pp.separar_xy(train_pp)
X_val, y_val = pp.separar_xy(val_pp)
X_test, y_test = pp.separar_xy(test_pp)
print(X_train.shape, X_val.shape, X_test.shape, "| NaN:", sum(X.isna().sum().sum() for X in (X_train, X_val, X_test)))

# todo junto: X_train, X_val, X_test, y_train, y_val, y_test = pp.run(df)

  -> [fit] train (848, 392)
  -> [fit] inválidos (nan): 125 valores, quedan 848 filas
  -> [fit] outliers (iqr, nan): 13334 valores, quedan 848 filas
  -> [fit] transformer yeo-johnson entrenado
  -> [fit] scaler robust entrenado
  -> [fit] imputador knn entrenado, 0 columnas descartadas por nulos
  -> [train] inicio (848, 392)
  -> [train] filas sin target eliminadas: 0
  -> [train] inválidos (nan): 125 valores, quedan 848 filas
  -> [train] outliers (iqr, nan): 13334 valores, quedan 848 filas
  -> [train] transformer yeo-johnson aplicado
  -> [train] scaler robust aplicado
  -> [train] faltantes (knn): 59211 valores imputados
  -> [train] fin (848, 390)
  -> [val] inicio (182, 392)
  -> [val] filas sin target eliminadas: 0
  -> [val] inválidos (nan): 12 valores, quedan 182 filas
  -> [val] outliers (iqr, nan): 6035 valores, quedan 182 filas
  -> [val] transformer yeo-johnson aplicado
  -> [val] scaler robust aplicado
  -> [val] faltantes (knn): 9358 valores imputados
  -> [val] fin (

### Selección de variables

Primero un **filtro** (score univariado contra el target, se queda con las `k` mejores) y sobre esas un **wrapper**:

| Paso | Config | Opciones |
|---|---|---|
| Filtro | `ConfigFiltro` | `pearson` / `spearman` (\|corr\|), `f_regression` (test F), `mutual_info` (no lineal); `k` |
| Wrapper | `ConfigWrapper` | `lasso` (`alpha`), `rfe`, `forward`, `backward` (`n_features`, `estimador` ridge / lgbm) |

El estudio de Optuna busca la mejor combinación **sin cross validation**: cada trial selecciona con train, entrena `MODELO_EVAL` con esas columnas y se evalúa con el MAE en **val**. Test no se mira. La mejor config se imprime lista para pegar en `src/main.py`.

In [ ]:
import optuna
from optuna.pruners import MedianPruner
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error

N_TRIALS = 100
MAX_EPOCH = 200

MODELO_EVAL = Ridge  # modelo con el que se puntúa cada selección
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objetivo(trial):
    filtro = ConfigFiltro(trial.suggest_categorical("filtro", ["pearson", "spearman", "f_regression", "mutual_info"]),
                          k=trial.suggest_int("k", 20, 390, step=10))
    metodo = trial.suggest_categorical("wrapper", ["lasso", "rfe", "forward", "backward"])
    if metodo == "lasso":
        wrapper = ConfigWrapper(metodo, alpha=trial.suggest_float("alpha", 1e-6, 3, log=True))
    else:
        wrapper = ConfigWrapper(metodo, n_features=trial.suggest_int("n_features", 20, 390),
                                estimador=trial.suggest_categorical("estimador", ["ridge", "lgbm"]))
    try:
        sel = SeleccionVariables(filtro, wrapper)
        Xtr = sel.fit_transform(X_train, y_train)
    except ValueError:  # p. ej. lasso con alpha alto que no deja columnas
        raise optuna.TrialPruned()
    trial.set_user_attr("n_columnas", Xtr.shape[1])
    
    for epoch in range(MAX_EPOCH):
        modelo = MODELO_EVAL().fit(Xtr, y_train)
        mae = mean_absolute_error(y_val, modelo.predict(sel.transform(X_val)))
        trial.report(mae, step=epoch)      # reporta mae de validación
        if trial.should_prune():
            raise optuna.TrialPruned()

    return mean_absolute_error(y_val, modelo.predict(sel.transform(X_val)))

if OPTIMIZAR:
    study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=RNG), pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=3))
    study.optimize(objetivo, 
                   n_trials=N_TRIALS, 
                   timeout=300, # 5 min
                   show_progress_bar=True, 
                   n_jobs=-1) # cambiar de -1 si no queres que te use todo el cpu
    display(study.trials_dataframe().sort_values("value").head(10))
    print(optuna.importance.get_param_importances(study))

  0%|          | 0/100 [00:00<?, ?it/s]

c:\Users\Arturo\miniconda3\envs\cervecera_env\Lib\site-packages\numpy\lib\_function_base_impl.py:3036: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\Arturo\miniconda3\envs\cervecera_env\Lib\site-packages\numpy\lib\_function_base_impl.py:3037: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
c:\Users\Arturo\miniconda3\envs\cervecera_env\Lib\site-packages\numpy\lib\_function_base_impl.py:3036: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\Arturo\miniconda3\envs\cervecera_env\Lib\site-packages\numpy\lib\_function_base_impl.py:3037: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
c:\Users\Arturo\miniconda3\envs\cervecera_env\Lib\site-packages\numpy\lib\_function_base_impl.py:3036: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\Arturo\miniconda3\envs\cervecera_env\Lib\site-packages\numpy\lib\_function_base_impl.py:3037: Runtime

In [ ]:
# mejor combinación lista para copiar a main.py (si no se optimizó, queda la de default)
if OPTIMIZAR:
    b = study.best_params
    filtro = ConfigFiltro(metodo=b["filtro"], k=b["k"])
    wrapper = (ConfigWrapper(metodo="lasso", alpha=b["alpha"]) if b["wrapper"] == "lasso" else
               ConfigWrapper(metodo=b["wrapper"], n_features=b["n_features"], estimador=b["estimador"]))
    print(f"MAE val: {study.best_value:.2f}", end=2 * chr(10))
    print(f"        filtro=ConfigFiltro(metodo={filtro.metodo!r}, k={filtro.k}),")
    print("        wrapper=ConfigWrapper(" + (f"metodo='lasso', alpha={wrapper.alpha:.6g}" if wrapper.metodo == "lasso" else
          f"metodo={wrapper.metodo!r}, n_features={wrapper.n_features}, estimador={wrapper.estimador!r}") + "),")
else:
    filtro, wrapper = ConfigFiltro(), ConfigWrapper()

sel = SeleccionVariables(filtro, wrapper, verbose=True)
X_train = sel.fit_transform(X_train, y_train)
X_val, X_test = sel.transform(X_val), sel.transform(X_test)
sel.seleccionadas_

  -> selección: 390 columnas -> filtro 60 -> wrapper 59


c:\Users\Arturo\miniconda3\envs\cervecera_env\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:842: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 5.602155e+09, tolerance: 5.538e+06
  model = cd_fast.enet_coordinate_descent(


['frio (kw)',
 'sala maq (kw)',
 'frio (kw) mm7',
 'frio (kw) mm14',
 'frio (kw) mm3',
 'servicios (kw)',
 'planta (kw) mm14',
 'planta (kw) mm7',
 'kw gral planta',
 'planta (kw)',
 'temp max (°c) x frio (kw)',
 'planta (kw) mm3',
 'frio (kw) lag1',
 'mes_cos',
 'kw trafo 11',
 'frio (kw) lag2',
 'frio (kw) lag7',
 'agua planta (hl)',
 'kw cond 5. 6 y 9',
 'planta de agua (hl)',
 'planta (kw) lag7',
 'hl cerveza envasada mm14',
 'planta (kw) lag1',
 'produccion (hl)',
 'meta et bodega',
 'meta et linea 2',
 'meta et servicios',
 'meta et linea 4',
 'meta et cocina',
 'meta et linea 5',
 'meta et linea 3',
 'planta (kw) lag2',
 'cumpl et servicios',
 'hl cerveza envasada mm7',
 'temp tq intermedio',
 'et bodega/hl',
 'cumpl et bodega',
 'et servicios / hl',
 'tot l3, l4 y planta de co2',
 'kw mycom 7',
 'temp max (°c) x hl cerveza envasada',
 'kw mycom 2',
 'kw trafo 10',
 'kw compresores aire',
 'aire producido (m3)',
 'totalizador sistema kaeser',
 'aire (kw)',
 'elaboracion (kw)',
 

## Resumen

In [ ]:
pd.DataFrame(pp.resumen()["pasos"])

,train,val,test
filas_inicio,848,182,182
invalidos_valores,125,12,9
outliers_valores,13334,6035,5457
transformer,yeo-johnson,yeo-johnson,yeo-johnson
scaler,robust,robust,robust
columnas_descartadas_por_nulos,0,0,0
faltantes_valores,59211,9358,8299
filas_fin,848,182,181


### Dataset final y linaje

In [ ]:
if GUARDAR_FINAL:
    guardar_dataset_final(pp, {"train": (X_train, y_train), "val": (X_val, y_val), "test": (X_test, y_test)},
                          script="preprocesamiento.ipynb", cfg_derivadas=cfg_derivadas, seleccion=sel)
{"X_train": X_train.shape, "X_val": X_val.shape, "X_test": X_test.shape}

{'X_train': (848, 59), 'X_val': (182, 59), 'X_test': (181, 59)}